# Intraday Probability Band Engine

## Backtest Research Notebook

This notebook is the main research and backtesting workspace for the probability band engine.

It loads historical intraday data, applies the reusable `src/` modules, and evaluates VWAP / TWAP reference lines, sigma bands, z-score zones, outcome labels, and empirical probability tables.

The project is structured so that:
- reusable engine logic lives in `src/`
- research, diagnostics, and charts live in this notebook
- live monitoring and trading workflow live in `notebooks/live_trading.ipynb`

---

## What this notebook does

1. Loads and inspects historical intraday data
2. Checks data quality and session structure
3. Computes the intraday reference line
4. Builds sigma bands and z-score zones
5. Extracts context variables
6. Labels historical outcomes
7. Calibrates empirical conditional probabilities
8. Runs backtest analysis and diagnostics
9. Produces charts, tables, and research outputs

### Section 0 — Configuration and Imports

In [ ]:
from src.config import CONFIG
from src.loaders import load_mt5_csv, load_mt5_live, load_tradingview_csv, load_generic_csv, assign_sessions

### Section 1 — Data Loading and Basic Inspection

In [ ]:
print("✅ Configuration loaded")
print(f"   Instrument : {CONFIG['instrument']}")
print(f"   Reference  : {CONFIG['reference_type']}")
print(f"   Vol method : {CONFIG['vol_method']}")
print(f"   Horizon    : {CONFIG['outcome_horizon_bars']} bars")
print(f"   Spread     : {CONFIG['spread_cost']} price units")
print(f"   Edge gap   : {CONFIG['edge_gap_threshold']:.0%} minimum")

In [ ]:
# ── Load data — change loader here if your source changes ──
df = load_mt5_csv(CONFIG['csv_path'])

# ── Basic inspection ──
print(f"\nDate range  : {df['datetime'].iloc[0]} → {df['datetime'].iloc[-1]}")
print(f"Total bars  : {len(df):,}")
print(f"Unique days : {df['session_date'].nunique()}")
print(f"\nColumn dtypes:")
print(df.dtypes)
print(f"\nFirst 3 rows:")
df.head(3)

### Section 2 — Data Quality Checks

In [ ]:
# ── Data quality checks ──
issues = []

zero_vol = (df['tick_volume'] <= 1.0).sum()
if zero_vol > len(df) * 0.05:
    issues.append(f"⚠️  {zero_vol:,} bars ({zero_vol/len(df):.1%}) have volume ≤ 1 — VWAP may be unreliable")

null_count = df.isnull().sum().sum()
if null_count > 0:
    issues.append(f"⚠️  {null_count} null values found — check loader output")

# Detect time gaps larger than expected bar duration
time_diffs = df['datetime'].diff().dropna()
median_diff = time_diffs.median()
large_gaps = (time_diffs > median_diff * 3).sum()
if large_gaps > 0:
    issues.append(f"⚠️  {large_gaps} time gaps > 3x bar duration (weekends/news expected)")

if issues:
    for i in issues:
        print(i)
else:
    print("✅ Data quality checks passed")

print(f"\nMedian bar duration: {median_diff}")
print(f"Volume stats:\n{df['tick_volume'].describe().round(2)}")